In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from rdkit import Chem

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

from sklearn.model_selection import train_test_split

c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
df = pd.read_csv(
    "../data/processed/bindingdb_clean.csv"
)

print(df.shape)

df.head()

(22232, 4)


,Ligand SMILES,Target Name,Ki (nM),pKi
0,CCC(c1ccccc1)c1c(O)c2ccccc2oc1=O,Dimer of Gag-Pol polyprotein [489-587],1000.0,6.000000
1,CCC(c1ccccc1)c1c(O)cc(CCc2ccccc2)oc1=O,Dimer of Gag-Pol polyprotein [489-587],500.0,6.301030
2,CCC(Cc1ccccc1)c1cc(O)c(C(CC)c2ccccc2)c(=O)o1,Dimer of Gag-Pol polyprotein [489-587],38.0,7.420216
3,Oc1c2CCCCCCc2oc(=O)c1C(C1CC1)c1ccccc1,Dimer of Gag-Pol polyprotein [489-587],15.0,7.823909
4,CCC(Cc1ccccc1)c1cc(O)c(C(CC)c2ccccc2)c(=O)o1,Dimer of Gag-Pol polyprotein [514-612],32.0,7.494850


small training subset

In [3]:
df = df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

print(df.shape)

(5000, 4)


smiles -> graph

In [4]:
def smiles_to_graph(
    smiles,
    target
):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = []

    for atom in mol.GetAtoms():

        x.append([
            atom.GetAtomicNum()
        ])

    edges = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edges.append([i, j])
        edges.append([j, i])

    if len(edges) == 0:
        return None

    return Data(
        x=torch.tensor(
            x,
            dtype=torch.float
        ),

        edge_index=torch.tensor(
            edges,
            dtype=torch.long
        ).t(),

        y=torch.tensor(
            [target],
            dtype=torch.float
        )
    )

build dataset

In [5]:
graphs = []

for _, row in df.iterrows():

    graph = smiles_to_graph(
        row["Ligand SMILES"],
        row["pKi"]
    )

    if graph is not None:
        graphs.append(graph)

print(
    "Graphs:",
    len(graphs)
)

Graphs: 4953


train/test split

In [6]:
train_graphs, test_graphs = train_test_split(
    graphs,
    test_size=0.2,
    random_state=42
)

train_loader = DataLoader(
    train_graphs,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_graphs,
    batch_size=64
)

print(
    len(train_graphs),
    len(test_graphs)
)

3962 991


GNN regressor

In [7]:
class AffinityGNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(1, 64)

        self.conv2 = GCNConv(64, 128)

        self.fc = nn.Linear(128, 1)

    def encode(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = global_mean_pool(
            x,
            batch
        )

        return x

    def forward(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.encode(
            x,
            edge_index,
            batch
        )

        return self.fc(x)

model

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AffinityGNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.MSELoss()

print(model)


AffinityGNN(
  (conv1): GCNConv(1, 64)
  (conv2): GCNConv(64, 128)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


Training loop

In [9]:
for epoch in range(10):

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        pred = model(
            batch.x,
            batch.edge_index,
            batch.batch
        )

        loss = criterion(
            pred.squeeze(),
            batch.y
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}:",
        total_loss / len(train_loader)
    )

Epoch 1: 15.361087749081273
Epoch 2: 4.043749355500744
Epoch 3: 4.020000053990271
Epoch 4: 3.976347173413923
Epoch 5: 3.9458587035056083
Epoch 6: 3.9330629033427082
Epoch 7: 3.8790059166569866
Epoch 8: 3.837829259134108
Epoch 9: 3.7940476498296185
Epoch 10: 3.7429896208547775


evaluation

In [10]:
model.eval()

losses = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(device)

        pred = model(
            batch.x,
            batch.edge_index,
            batch.batch
        )

        loss = criterion(
            pred.squeeze(),
            batch.y
        )

        losses.append(
            loss.item()
        )

print(
    "Test MSE:",
    np.mean(losses)
)

Test MSE: 3.085891753435135


In [11]:
torch.save(
    model.state_dict(),
    "../models/checkpoints/pharmagpt_v0_7.pt"
)

print("Model Saved")

Model Saved
